# ICMI Corpus — Parsing GAT-2 Transcripts with Python
### Notebook 1 of 3: From PDF to Structured Data

This notebook reads GAT-2 transcript files (PDF or plain text) and converts them
into structured pandas DataFrames. Each step is explained for readers without
programming experience.

**Output:** One CSV per transcript in `csv_export/` — input for Notebook 2.

---


## Part 1 — Setup

We load the libraries we need. `pdfplumber` extracts text from PDF files.
`pandas` creates and manipulates data tables.
`re` provides regular expressions for pattern matching.


In [89]:
import subprocess
subprocess.run(["pip", "install", "pdfplumber"], capture_output=True)

import re
import glob
import pandas as pd
import pdfplumber
from pathlib import Path
from collections import Counter

print("All libraries loaded.")


All libraries loaded.


---
## Part 2 — Reading Text from Files

The transcription files use the `.pdf` extension but come in two variants:

1. **True PDF files** — text is embedded in a complex layout. Identifiable
   because the file begins with the bytes `%PDF`. We use `pdfplumber` to extract the text.
2. **Plain text files with a `.pdf` extension** — the content is raw text;
   the extension is misleading. These can be read directly.

The function below detects both variants automatically.

A special challenge arises in some files: GAT-2 uses a **score-like layout**
where the speaker ID sits on the left and the utterance text on the right,
at different horizontal positions on the same line. Standard PDF extraction
reads these as separate lines, breaking the speaker–text pairing.
The solution uses `pdfplumber`'s word-level position data (X/Y coordinates)
to detect this layout and merge the two columns back together.


In [90]:
def _extract_page(page):
    """
    Extracts text from one PDF page.
    Detects two-column layouts (speaker ID left, text right)
    and merges them correctly using word position data.
    """
    from collections import defaultdict

    words = page.extract_words()
    if not words:
        return page.extract_text() or ""

    right_col = [w for w in words if w["x0"] >= 250]
    left_col  = [w for w in words if w["x0"] <  250]

    if not right_col:
        return page.extract_text() or ""

    # Group words into lines by Y position (tolerance: 3 points)
    Y_TOL = 3
    left_lines  = defaultdict(list)
    right_lines = defaultdict(list)

    for w in left_col:
        left_lines[round(w["top"] / Y_TOL) * Y_TOL].append(w)
    for w in right_col:
        right_lines[round(w["top"] / Y_TOL) * Y_TOL].append(w)

    all_y = sorted(set(list(left_lines.keys()) + list(right_lines.keys())))
    result = {}
    last_left_y = None

    for y in all_y:
        if y in left_lines:
            text = " ".join(w["text"] for w in sorted(left_lines[y], key=lambda w: w["x0"]))
            result[y] = text
            last_left_y = y
        if y in right_lines:
            text = " ".join(w["text"] for w in sorted(right_lines[y], key=lambda w: w["x0"]))
            if last_left_y is not None:
                result[last_left_y] = result[last_left_y] + " " + text
            else:
                result[y] = text

    return "\n".join(result[y] for y in sorted(result.keys()))


def read_raw_text(filepath):
    """
    Reads raw text from a transcription file.
    Handles both true PDFs and plain-text files automatically.
    """
    with open(filepath, "rb") as f:
        header = f.read(4)

    if header == b"%PDF":
        pages = []
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                text = _extract_page(page)
                if text:
                    pages.append(text)
        return "\n".join(pages)
    else:
        return open(filepath, encoding="utf-8", errors="replace").read()


# Find all transcription files (filename starts with a year)
all_files = sorted(glob.glob("20*.pdf"))
print(f"{len(all_files)} transcription files found:")
for f in all_files:
    with open(f, "rb") as fp:
        is_pdf = fp.read(4) == b"%PDF"
    print(f"  {'PDF    ' if is_pdf else 'Text   '}  {f}")


13 transcription files found:
  PDF      2012_Uberaba_Alemaes_Sueca1.pdf
  PDF      2013_Muenster_Alemaes1-Parte1.pdf
  PDF      2013_Muenster_Alemaes1-Parte2.pdf
  PDF      2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2.pdf
  PDF      2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3.pdf
  PDF      2014_Muenster_Alemaes2-Parte1.pdf
  PDF      2014_Muenster_Alemaes2-Parte2.pdf
  PDF      2014_Muenster_Brasileiros2-Parte1.pdf
  PDF      2014_Muenster_Brasileiros2-Parte2.pdf
  PDF      2014_Muenster_Brasileiros2-Parte3.pdf
  PDF      2015_BeloHorizonte_Alemas_Brasileiras_Heimat1.pdf
  PDF      2015_BeloHorizonte_Brasileiros_Assembleia1.pdf
  PDF      2016_BeloHorizonte_Brasileiros_Assembleia2.pdf


---
## Part 3 — Understanding the File Structure

Each transcription file is divided into three sections:

```
HEADER          →  Metadata: project name, location, date, participants ...
SPEAKERTABLE    →  Speaker IDs with L1, L2, sex, occupation ...
TURNS           →  The actual conversation data: [1] through [N]
```

Within the turns section, four line types occur:

| Type | Example |
|---|---|
| Turn number | `[7]` |
| Timestamp | `18 [00:30.0] 19 [00:31.0]` |
| Verbal utterance | `A1 [v] ich verstehe nicht VIEL` |
| Pause / non-verbal | `[nv] (1.2)` |


In [91]:
def classify_line(line):
    """
    Assigns a line to one of six categories based on its form.
    Uses regular expressions to match patterns at the start of the line.
    """
    z = line.strip()
    if z == "":
        return "empty"
    if re.match(r"^\[\d+\]$", z):
        return "turn_number"
    if re.match(r"^(\.\. )?\d+\s*\[", z):
        return "timestamp"
    if re.match(r"^[A-Za-z][A-Za-z0-9]*(?:\s+\S+)?\s+\[v\]", z):
        return "verbal"
    if re.match(r"^[A-Za-z][A-Za-z0-9]*(?:\s+\S+)?\s+\[nv\]", z):
        return "nonverbal_speaker"
    if re.match(r"^\[nv\]", z):
        return "nonverbal_solo"
    return "other"


# Test: count line types in the first file
sample_file = all_files[0]
raw_text = read_raw_text(sample_file)
lines = raw_text.splitlines()

counts = Counter(classify_line(l) for l in lines)
print(f"Line types in: {Path(sample_file).name}\n")
for cat, n in counts.most_common():
    print(f"  {cat:20s}  {n:5d}")


Line types in: 2012_Uberaba_Alemaes_Sueca1.pdf

  verbal                 2924
  other                  2191
  turn_number            1618
  timestamp              1545


---
## Part 4 — Parsing the Three Sections

We write one function for each section.
*Parsing* means systematically breaking structured text into its components.


### 4.1 Header

In [92]:
def parse_header(header_lines):
    """
    Parses the metadata header into a dict: {field_name: value}.
    Handles multi-line values (e.g. long project titles).
    """
    header = {}
    current_key = None
    buffer = []

    for line in header_lines:
        line = line.strip()
        if not line:
            continue
        m = re.match(r"^([^:]+):\s*(.*)$", line)
        if m:
            if current_key:
                header[current_key] = " ".join(buffer).strip()
            current_key = m.group(1).strip()
            buffer = [m.group(2).strip()]
        else:
            if current_key:
                buffer.append(line)

    if current_key:
        header[current_key] = " ".join(buffer).strip()
    return header


### 4.2 Occupation from Header

Some files list occupations in the header in the format
`Professoras (A2, B1) e Estudante (A1)`.
We extract this and map Portuguese terms to unified English labels.


In [93]:
def parse_occupations(header):
    """
    Extracts speaker -> occupation from the Profissão line in the header.
    Returns a dict: {speaker_id: occupation_label}
    """
    mapping = {
        "professoras": "prof", "professores": "prof",
        "professor":   "prof", "professora":  "prof",
        "estudante":   "stud", "estudantes":  "stud",
        "student":     "stud", "studentin":   "stud",
        "pastor":      "past", "pastora":     "past",
    }
    occupation_line = header.get("Profissão dos participantes", "")
    result = {}
    for match in re.finditer(r"(\w+)\s*\(([^)]+)\)", occupation_line):
        label = mapping.get(match.group(1).lower(), match.group(1).lower())
        for spk in match.group(2).split(","):
            result[spk.strip().split(":")[0].strip()] = label
    return result


### 4.3 Speaker Table

The speaker table lists all participants with their attributes.
Speaker IDs vary across files: `A1`, `D2m`, `Operador 1`, `MGP` — we use
the full line (not just the first word) as the ID.
We keep only the columns that are consistent across all files:
`sex`, `l1`, `l2`, `occupation`.


In [94]:
def parse_speaker_table(speaker_lines, occupations_from_header):
    """
    Parses the speaker table.
    Returns a dict: {speaker_id: {sex, l1, l2, occupation}}
    """
    field_map = {
        "sex":       "sex",
        "l1":        "l1",
        "l2":        "l2",
        "profissão": "occupation",
    }
    speakers = {}
    current = None

    for line in speaker_lines:
        z = line.strip()
        if not z or z.lower() == "speakertable":
            continue
        # Speaker ID: short line, no colon, starts with uppercase
        if len(z) < 40 and ":" not in z and re.match(r"^[A-Z]", z):
            current = z
            speakers[z] = {}
        elif current:
            m = re.match(r"^([\w\s\u00c0-\u024f]+):\s*(.*)$", z)
            if m:
                key_raw = m.group(1).strip().lower().replace(" ", "_")
                key = field_map.get(key_raw)
                if key:
                    speakers[current][key] = m.group(2).strip() or "unknown"

    # Fill missing fields and add occupation from header if not already set
    standard_fields = ["sex", "l1", "l2", "occupation"]
    for spk in speakers:
        if "occupation" not in speakers[spk]:
            speakers[spk]["occupation"] = occupations_from_header.get(spk, "unknown")
        for field in standard_fields:
            if field not in speakers[spk]:
                speakers[spk][field] = "unknown"

    return speakers


### 4.4 Timestamps

In [95]:
def parse_timestamp(line):
    """
    Extracts segment numbers and time values from a timestamp line.
    Returns: (start_time_str, end_time_str, list_of_segment_numbers)
    Example: '5 [00:05.0] 6 [00:06.0]' -> ('00:05.0', '00:06.0', [5, 6])
    """
    pairs = re.findall(r"(\d+)\s*\[(\d+:\d+\.\d+)\]", line)
    if not pairs:
        return None, None, []
    return pairs[0][1], pairs[-1][1], [int(p[0]) for p in pairs]


### 4.5 Turns

The core parser. We read the turns section line by line and build
one record per utterance.

Note: lines starting with `..` are continuations of the previous turn's
timestamp. The last segment number of the previous turn is carried over.

`[nv]` lines mark phases with no verbal activity (silence).
They differ from GAT-2-encoded pauses within utterances
(e.g. `(.)`, `(--)`) which appear inside `[v]` lines.


In [96]:
def parse_turns(turn_lines, known_speakers=None):
    """
    Parses all turns into a list of dicts — one utterance per entry.
    Fields: turn_id, segment_start, segment_end, segments,
            speaker_id, channel, text

    known_speakers: set of speaker IDs from the speaker table.
    Required for files that omit the [v] marker.
    """
    known_speakers = known_speakers or set()
    records         = []
    current_turn    = None
    current_start   = None
    current_end     = None
    current_segments = []
    last_segment_nr  = None

    for line in turn_lines:
        cat = classify_line(line)

        if cat == "turn_number":
            current_turn     = int(re.search(r"\d+", line).group())
            current_start    = None
            current_end      = None
            current_segments = []

        elif cat == "timestamp":
            start, end, segments = parse_timestamp(line)
            if line.strip().startswith("..") and last_segment_nr is not None:
                segments = [last_segment_nr] + segments
            if current_start is None:
                current_start = start
            current_end = end
            current_segments.extend(segments)
            if segments:
                last_segment_nr = segments[-1]

        elif cat == "verbal":
            m = re.match(r"^(.+?)\s+\[v\]\s*(.*)", line.strip())
            if m:
                records.append({
                    "turn_id":       current_turn,
                    "segment_start": current_start,
                    "segment_end":   current_end,
                    "segments":      current_segments.copy(),
                    "speaker_id":    m.group(1).strip(),
                    "channel":       "v",
                    "text":          m.group(2).strip(),
                })

        elif cat == "nonverbal_solo":
            records.append({
                "turn_id":       current_turn,
                "segment_start": current_start,
                "segment_end":   current_end,
                "segments":      current_segments.copy(),
                "speaker_id":    "[nv]",
                "channel":       "nv",
                "text":          line.strip(),
            })

        elif cat == "other" and current_turn is not None and known_speakers:
            # Alternative format without [v] marker
            parts = line.strip().split(None, 1)
            if parts and parts[0] in known_speakers:
                records.append({
                    "turn_id":       current_turn,
                    "segment_start": current_start,
                    "segment_end":   current_end,
                    "segments":      current_segments.copy(),
                    "speaker_id":    parts[0],
                    "channel":       "v",
                    "text":          parts[1].strip() if len(parts) > 1 else "",
                })

    return records


---
## Part 5 — Main Function: File to DataFrame

All parser functions are combined into one function.
It takes a file path and returns a complete, annotated DataFrame.


In [97]:
def file_to_dataframe(filepath):
    """
    Reads a GAT-2 transcription file (PDF or plain text)
    and returns a pandas DataFrame.

    Columns: turn_id, segment_start, segment_end, segments,
             speaker_id, channel, text, file,
             sex, l1, l2, occupation
    """
    # 1. Read text
    raw_text = read_raw_text(filepath)
    lines    = raw_text.splitlines()

    # 2. Split into sections
    idx_st = next((i for i, l in enumerate(lines)
                   if "speakertable" in l.lower()), None)
    idx_t1 = next((i for i, l in enumerate(lines)
                   if classify_line(l) == "turn_number"), None)

    if idx_st is None or idx_t1 is None:
        print(f"  WARNING: structure not recognized — {Path(filepath).name}")
        return None

    header_lines  = lines[:idx_st]
    speaker_lines = lines[idx_st:idx_t1]
    turn_lines    = lines[idx_t1:]

    # 3. Parse metadata
    header      = parse_header(header_lines)
    occupations = parse_occupations(header)
    speakers    = parse_speaker_table(speaker_lines, occupations)

    # 4. Parse turns
    records = parse_turns(turn_lines, known_speakers=set(speakers.keys()))
    df = pd.DataFrame(records)
    df["file"] = Path(filepath).stem

    # 5. Merge speaker metadata
    if speakers and len(df) > 0 and "speaker_id" in df.columns:
        meta = pd.DataFrame(speakers).T
        meta.index.name = "speaker_id"
        meta = meta.reset_index()
        df = df.merge(meta, on="speaker_id", how="left")

    # 6. Enforce column order — drop age and education
    core_columns = ["turn_id", "segment_start", "segment_end", "segments",
                    "speaker_id", "channel", "text", "file",
                    "sex", "l1", "l2", "occupation"]
    present = [c for c in core_columns if c in df.columns]
    df = df[present]

    return df


---
## Part 6 — Read All Files


In [98]:
dataframes = {}

print("Reading all transcription files...\n")
for filepath in all_files:
    name = Path(filepath).stem
    df   = file_to_dataframe(filepath)

    if df is not None:
        dataframes[name] = df
        n_turns    = df["turn_id"].nunique()
        n_segments = df["segments"].explode().nunique()
        n_speakers = df[df["channel"] == "v"]["speaker_id"].nunique()
        print(f"{name}")
        print(f"  Turns: {n_turns}  |  Segments: {n_segments}  "
              f"|  Speakers: {n_speakers}  |  Utterances: {len(df)}")
    print()

print(f"Total files loaded: {len(dataframes)}")


Reading all transcription files...

2012_Uberaba_Alemaes_Sueca1
  Turns: 1618  |  Segments: 2741  |  Speakers: 7  |  Utterances: 2927

2013_Muenster_Alemaes1-Parte1
  Turns: 1128  |  Segments: 2769  |  Speakers: 6  |  Utterances: 2512

2013_Muenster_Alemaes1-Parte2
  Turns: 491  |  Segments: 1152  |  Speakers: 6  |  Utterances: 1112

2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2
  Turns: 215  |  Segments: 294  |  Speakers: 8  |  Utterances: 395

2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3
  Turns: 239  |  Segments: 524  |  Speakers: 9  |  Utterances: 395

2014_Muenster_Alemaes2-Parte1
  Turns: 918  |  Segments: 1915  |  Speakers: 5  |  Utterances: 1859

2014_Muenster_Alemaes2-Parte2
  Turns: 211  |  Segments: 519  |  Speakers: 5  |  Utterances: 496

2014_Muenster_Brasileiros2-Parte1
  Turns: 761  |  Segments: 1263  |  Speakers: 6  |  Utterances: 1474

2014_Muenster_Brasileiros2-Parte2
  Turns: 839  |  Segments: 1403  |  Speakers: 5  |  Utterances: 1429

---
## Part 7 — Overview

Two summary views of the parsed data.


### 7.1 nonverbal_solo — Phases Without Any Speaker

`[nv]` lines mark phases in which no one is speaking.
They do not appear in all transcripts: some projects encode pauses
directly within utterances as `(.)`, `(--)`, or `(1.2)` inside `[v]` lines.
The analysis of those in-utterance pauses will be carried out in the analysis notebook.


In [99]:
nv_solo = pd.concat(
    [df_i[df_i["channel"] == "nv"].assign(file=name)
     for name, df_i in dataframes.items()],
    ignore_index=True
)

print(f"[nv] lines total (all files): {len(nv_solo)}")

if len(nv_solo) == 0:
    print("No [nv] lines found in the corpus.")
else:
    print(f"Found in: {nv_solo['file'].unique().tolist()}")
    nv_solo["duration_sec"] = nv_solo["text"].apply(
        lambda t: float(m.group(1)) if (m := re.search(r"\((\d+\.?\d*)s?\)", t)) else None
    ).astype(float)
    print("\nDuration (seconds):")
    print(nv_solo["duration_sec"].describe().round(2))


[nv] lines total (all files): 58
Found in: ['2015_BeloHorizonte_Alemas_Brasileiras_Heimat1']

Duration (seconds):
count    54.00
mean      0.96
std       0.54
min       0.30
25%       0.60
50%       0.82
75%       1.20
max       3.50
Name: duration_sec, dtype: float64


### 7.2 Overview of All Files

In [100]:
overview = []
for name, df_i in dataframes.items():
    verbal_i = df_i[df_i["channel"] == "v"]
    overview.append({
        "file":        name,
        "turns":       df_i["turn_id"].nunique(),
        "segments":    df_i["segments"].explode().nunique(),
        "utterances":  len(verbal_i),
        "nv_solo":     len(df_i[df_i["channel"] == "nv"]),
        "speakers":    verbal_i["speaker_id"].nunique(),
    })

overview_df = pd.DataFrame(overview).set_index("file")
print("Overview of all transcription files:\n")
print(overview_df.to_string())


Overview of all transcription files:

                                                               turns  segments  utterances  nv_solo  speakers
file                                                                                                         
2012_Uberaba_Alemaes_Sueca1                                     1618      2741        2927        0         7
2013_Muenster_Alemaes1-Parte1                                   1128      2769        2512        0         6
2013_Muenster_Alemaes1-Parte2                                    491      1152        1112        0         6
2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2    215       294         395        0         8
2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3    239       524         395        0         9
2014_Muenster_Alemaes2-Parte1                                    918      1915        1859        0         5
2014_Muenster_Alemaes2-Parte2                                    211       519    

### Part 7.3 — Map Speaker Metadata from External CSV

The speaker metadata parsed directly from the transcription files
may be incomplete or inconsistently labeled across files.
We therefore load a curated, corpus-wide metadata table
(`icmi_speaker_metadata.csv`) and map it onto all DataFrames.
This overwrites any metadata already present, since the CSV
is the corrected and authoritative source for all speaker attributes.

In [101]:
# Load speaker metadata from external CSV and map onto all DataFrames
# This overwrites any metadata already parsed from the speaker tables,
# since the CSV is the corrected, authoritative source.

metadata = pd.read_csv("icmi_speaker_metadata.csv", sep=";")
metadata = metadata[["speaker_id", "sex", "l1", "l2", "occupation"]].drop_duplicates(subset="speaker_id")

for name in dataframes:
    df_i = dataframes[name]

    # Drop existing metadata columns
    drop_cols = [c for c in ["sex", "l1", "l2", "occupation"] if c in df_i.columns]
    df_i = df_i.drop(columns=drop_cols)

    # Merge on speaker_id only
    df_i = df_i.merge(metadata, on="speaker_id", how="left")

    # Re-enforce column order
    core_columns = ["turn_id", "segment_start", "segment_end", "segments",
                    "speaker_id", "channel", "text", "file",
                    "sex", "l1", "l2", "occupation"]
    present = [c for c in core_columns if c in df_i.columns]
    dataframes[name] = df_i[present]

print("Metadata mapped from icmi_speaker_metadata.csv.")
print()
example = list(dataframes.keys())[0]
print(f"Example — {example}:")
print(dataframes[example][["speaker_id","sex","l1","l2","occupation"]].drop_duplicates().to_string())

Metadata mapped from icmi_speaker_metadata.csv.

Example — 2012_Uberaba_Alemaes_Sueca1:
        speaker_id sex       l1        l2 occupation
0               A2   m      ger  eng; por    student
1       Operador 1   f  unknown   unknown    unknown
2       Operador 2   f  unknown   unknown    unknown
6      Transcritor   u  unknown   unknown    unknown
14              S1   f      swe  eng; por    student
32   Representante   m  unknown   unknown    unknown
280             A1   m      ger  eng; por    student


In [102]:
# Normalize speaker IDs: strip gender suffix and location from Münster-style IDs
# 'D1w Warschau' -> 'D1', 'D2m Barcelona' -> 'D2', 'B1' -> 'B1' (unchanged)

def normalize_speaker_id(raw_id):
    if not isinstance(raw_id, str):
        return raw_id
    m = re.match(r"^([A-Z]+\d+)[wm]?\s*(?:\S+)?$", raw_id)
    return m.group(1) if m else raw_id

for name in dataframes:
    dataframes[name]["speaker_id"] = dataframes[name]["speaker_id"].apply(normalize_speaker_id)

# Verify
example = list(dataframes.keys())[1]  # Münster file
print(f"Speaker IDs after normalization — {example}:")
print(dataframes[example]["speaker_id"].unique().tolist())

Speaker IDs after normalization — 2013_Muenster_Alemaes1-Parte1:
['D2', 'D4', 'D5', 'D3', 'D1', 'Gruppe']


---
## Part 8 — Export as CSV

Each DataFrame is saved as a CSV file.
The encoding `utf-8-sig` ensures that special characters (umlauts, accents)
display correctly in Excel.


In [103]:
import os

os.makedirs("csv_export", exist_ok=True)

for name, df_i in dataframes.items():
    output_path = f"csv_export/{name}.csv"
    df_i.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {output_path}  ({len(df_i)} rows, {len(df_i.columns)} columns)")

print(f"\nAll {len(dataframes)} files exported.")


Saved: csv_export/2012_Uberaba_Alemaes_Sueca1.csv  (2927 rows, 12 columns)
Saved: csv_export/2013_Muenster_Alemaes1-Parte1.csv  (2512 rows, 12 columns)
Saved: csv_export/2013_Muenster_Alemaes1-Parte2.csv  (1112 rows, 12 columns)
Saved: csv_export/2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2.csv  (395 rows, 12 columns)
Saved: csv_export/2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3.csv  (395 rows, 12 columns)
Saved: csv_export/2014_Muenster_Alemaes2-Parte1.csv  (1859 rows, 12 columns)
Saved: csv_export/2014_Muenster_Alemaes2-Parte2.csv  (496 rows, 12 columns)
Saved: csv_export/2014_Muenster_Brasileiros2-Parte1.csv  (1474 rows, 12 columns)
Saved: csv_export/2014_Muenster_Brasileiros2-Parte2.csv  (1429 rows, 12 columns)
Saved: csv_export/2014_Muenster_Brasileiros2-Parte3.csv  (1247 rows, 12 columns)
Saved: csv_export/2015_BeloHorizonte_Alemas_Brasileiras_Heimat1.csv  (2103 rows, 12 columns)
Saved: csv_export/2015_BeloHorizonte_Brasileiros_Assembleia1.csv  

---
## Outlook

The exported CSV files serve as input for the two subsequent notebooks:

- **Notebook 2** — Quantitative analysis of GAT-2 annotations:
  prosodic features, pauses, overlaps, speaker contributions
- **Notebook 3** — Lexical and collocational analysis with spaCy

Further possibilities:
- Visualizations with `matplotlib` or `seaborn`
- Statistical tests with `scipy` or `statsmodels`
- Import into R, Excel, or ELAN


In [104]:
import pandas as pd
df = pd.read_csv("csv_export/2013_Muenster_Alemaes1-Parte1.csv")
print(df["speaker_id"].unique().tolist())

['D2', 'D4', 'D5', 'D3', 'D1', 'Gruppe']


In [105]:
import glob, os
print("Working directory:", os.getcwd())
print("Files found:")
for f in sorted(glob.glob("20*.pdf")):
    print(" ", f)

Working directory: /Users/friederikeschulz/Library/Mobile Documents/com~apple~CloudDocs/Uni Potsdam/EPICS2026_Sevilla/PANDAS_Publikation
Files found:
  2012_Uberaba_Alemaes_Sueca1.pdf
  2013_Muenster_Alemaes1-Parte1.pdf
  2013_Muenster_Alemaes1-Parte2.pdf
  2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2.pdf
  2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3.pdf
  2014_Muenster_Alemaes2-Parte1.pdf
  2014_Muenster_Alemaes2-Parte2.pdf
  2014_Muenster_Brasileiros2-Parte1.pdf
  2014_Muenster_Brasileiros2-Parte2.pdf
  2014_Muenster_Brasileiros2-Parte3.pdf
  2015_BeloHorizonte_Alemas_Brasileiras_Heimat1.pdf
  2015_BeloHorizonte_Brasileiros_Assembleia1.pdf
  2016_BeloHorizonte_Brasileiros_Assembleia2.pdf
